# ELIZA kontra LLM — gdzie jest „rozumienie"?

**Podstawy AI · Laboratorium 2**

Na poprzednich zajęciach zbudowałeś ELIZĘ i znalazłeś miejsca, w których
jej złudzenie pęka. Dziś te same zdania trafią do dużego modelu językowego
(LLM) i sprawdzimy: **które pęknięcia LLM naprawia, a które zostają?**

**Plan zajęć**

| Sekcja | Co robisz |
|---|---|
| 1. Jak rozmawia się z LLM przez API | czytasz (5 min) |
| 2. Przygotowanie | klucz, klient, test połączenia |
| 3. Twoja ELIZA wraca | wklejasz kod z Lab 1 |
| 4. Ta sama rozmowa, dwa silniki | Zadanie 8 |
| 5. Łamiące zdania — druga runda | Zadanie 9 |
| 6. Pamięć to lista wiadomości | Zadanie 10 |
| 7. Rola to tekst | Zadanie 11 |
| 8. Refleksja | Zadanie 12 |

**Potrzebujesz:** klucza Gemini API (instrukcję dostałeś przed zajęciami)
i notebooka z Lab 1.

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 1. Jak rozmawia się z LLM przez API

**Czym jest LLM (w pięciu zdaniach).** Duży model językowy to sieć neuronowa
wytrenowana na ogromnych zbiorach tekstu do jednego zadania: przewidywania,
jaki fragment tekstu (*token*) powinien pojawić się dalej. Odpowiedź powstaje
token po tokenie — model za każdym razem wybiera kolejny fragment, a wybór
jest częściowo losowy (*próbkowanie*). Nie ma tu reguł pisanych ręcznie jak
w ELIZIE: wszystko, co model „umie", jest zapisane w miliardach wyuczonych
parametrów. To, czy i w jakim sensie model *rozumie* tekst, jest przedmiotem
sporu — i właśnie dlatego porównujemy go dziś z ELIZĄ.

**Zapytanie do API.** Rozmawiamy z modelem, wysyłając mu **listę wiadomości**.
Każda wiadomość ma rolę:

| Rola | Kto mówi | Przykład |
|---|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać | „Jesteś terapeutą. Odpowiadaj krótko." |
| `user` | użytkownik | „Czuję się samotny." |
| `assistant` | wcześniejsze odpowiedzi modelu | „Od kiedy tak się czujesz?" |

```python
messages = [
    {"role": "system", "content": "Jesteś terapeutą..."},
    {"role": "user",   "content": "Czuję się samotny."},
]
```

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 10 okaże się ważniejszy, niż wygląda.

**Biblioteka.** Użyjemy biblioteki `openai`. Mimo nazwy nie łączymy się
z OpenAI: Gemini udostępnia *zgodny* interfejs, więc wystarczy podać inny
adres serwera (`base_url`) i swój klucz.

## 2. Przygotowanie

Klucz podajesz przez `getpass` — nie pojawi się na ekranie ani w zapisanym
notebooku. **Nigdy nie wklejaj klucza wprost do kodu**: notebook oddajesz
prowadzącemu, a klucz to dostęp do Twojego konta.

Jeśli pracujesz w Google Colab, możesz zamiast tego zapisać klucz
w *Secrets* (ikona klucza po lewej) pod nazwą `GEMINI_API_KEY`.

In [ ]:
# !pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import os
import time
from getpass import getpass
from openai import OpenAI

try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                       # lokalnie: zmienna środowiskowa albo getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Wklej klucz Gemini API: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.5-flash-lite"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "minimal"             # ogranicza „myślenie" modelu -> szybciej i taniej;
                                  # jeśli dostajesz błąd 400 dotyczący reasoning_effort, ustaw None
print("Klient gotowy. Model:", MODEL)

Klient gotowy. Model: gemini-3.5-flash-lite


In [ ]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


Poniższa funkcja `czat` wysyła listę wiadomości i zwraca tekst odpowiedzi.
Obsługuje też błąd **429** (przekroczony limit zapytań na minutę) — wtedy
odczekuje chwilę i próbuje ponownie. Nie musisz jej modyfikować.

In [ ]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

Przykro mi, że tak się czujesz. Chcesz o tym porozmawiać?


**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 3. Twoja ELIZA wraca

Wklej poniżej **swoje** komórki z Lab 1: `normalizuj`, `odbicia`, `odbij`,
`reguly` (z regułą awaryjną) i `odpowiedz`. Porównujemy **Twoją** ELIZĘ.

Jeśli Twoja wersja nie działa, uruchom komórkę referencyjną pod spodem.

In [ ]:
# === Wklej tutaj swoją ELIZĘ z Lab 1 ===
import re
import random

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]

def normalizuj(tekst):
    # małe litery, bez interpunkcji — patrz sekcja 2
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

print("Gotowe. Zdań testowych:", len(TESTY))

odbicia = {
    # pacjent -> terapeuta
    "jestem": "jesteś",
    "moja": "twoja",
    "mnie": "ciebie",
    "mój":"twój",
    "moje":"twoje",
    "mam":"masz",
    "mi":"ci",
    "ja":"ty",
    "nasze":"wasze",
    "mojej":"twojej",
    "moją":"twoją",
    "moich":"twoich",
    # terapeuta -> pacjent
    "ty":"ja",
    "ci":"mi",
    # TODO (Zadanie 1): dopisz min. 6 par, np. mój/twój, moje/twoje, mam/masz,
    #                   mi/ci, ja/ty ... oraz min. 2 w kierunku odwrotnym (ci -> mi ...)
}

def odbij(fragment):
    slowa = fragment.lower().split()
    # TODO (Zadanie 1): zamień każde słowo na jego odbicie, jeśli jest w słowniku,
    #                   w przeciwnym razie zostaw bez zmian.
    return " ".join(odbicia.get(s,s) for s in slowa)   # <- uzupełnij

# (wzorzec, [szablony]) — dopasowujemy na tekście po normalizuj()
reguly = [
    (r"potrzebuję (.*)",
        ["Dlaczego potrzebujesz {0}?", "Czy naprawdę pomogłoby ci {0}?"]),
    (r"czuję się (.*)",
        ["Od jak dawna czujesz się {0}?", "Czy często tak się czujesz?"]),
    (r"\b(matka|ojciec|rodzina|rodzinie|siostra|brat)\b",
        ["Opowiedz mi więcej o swojej rodzinie.", "Jak układają się te relacje?"]),
    # TODO (Zadanie 2): dodaj min. 4 własne reguły
    (r"odczuwam (.*)",
        ["Dlaczego odczuwasz {0}?", "Czy często to odczuwasz?"]),
    (r"chcę (.*)",
        ["Dlaczego chcesz {0}?", "Z jakiego powodu chcesz {0}?"]),
    (r"jestem (.*)",
        ["Od jak dawna jesteś {0}?", "Jak to jest być {0}?"]),
    (r"myślę (.*)",
        ["Dlaczego myślisz że {0}?", "Co sprawia że myślisz że {0}?"]),

]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            # TODO (Zadanie 3a): odbij zaimki w każdej przechwyconej grupie
            grupy = [odbij(g) for g in m.groups()]      # <- uzupełnij
            # TODO (Zadanie 3b): wylosuj szablon i wstaw grupy w miejsce {0}, {1}...
            return random.choice(szablony).format(*grupy)                              # <- uzupełnij
    return None   # brak dopasowania — regułę awaryjną dodasz w Zadaniu 4

Gotowe. Zdań testowych: 6


In [ ]:
# === Referencyjna (uproszczona) ELIZA — tylko jeśli Twoja nie działa ===
import re, random

def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

odbicia = {"jestem": "jesteś", "moja": "twoja", "mój": "twój", "moje": "twoje",
           "mnie": "ciebie", "mi": "ci", "ja": "ty", "ci": "mi", "tobie": "mnie"}

def odbij(fragment):
    return " ".join(odbicia.get(s, s) for s in fragment.lower().split())

reguly = [
    (r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
    (r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
    (r"boję się (.*)",   ["Dlaczego boisz się {0}?"]),
    (r"\b(matka|ojciec|rodzina|brat|siostra)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
    (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),
]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            return random.choice(szablony).format(*[odbij(g) for g in m.groups()])

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]
print(odpowiedz("Czuję się dzisiaj samotny."))

## 4. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 8
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Pod spodem zapisz **3 konkretne różnice**, które widzisz. Nie oceniaj,
   który jest „lepszy" — opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [ ]:
wyniki = []
for zdanie in TESTY:
    odp_eliza = odpowiedz(zdanie)
    odp_llm = zapytaj_llm(zdanie)                     # <- uzupełnij (Zadanie 8)
    wyniki.append((zdanie, odp_eliza, odp_llm))
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

> Potrzebuję, żeby moja rodzina mnie słuchała.
  ELIZA: Czy naprawdę pomogłoby ci żeby twoja rodzina ciebie słuchała?
  LLM:   To naturalna potrzeba, każdy chce być usłyszany i zrozumiany przez najbliższych. Co dokładnie czujesz, kiedy czujesz się przez nich ignorowany?

> Czuję się dzisiaj samotny.
  ELIZA: Od jak dawna czujesz się dzisiaj samotny?
  LLM:   Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?

> Mój ojciec ciągle mnie krytykuje.
  ELIZA: Jak układają się te relacje?
  LLM:   To musi być dla Ciebie bardzo bolesne i wyczerpujące. Co dokładnie zazwyczaj mówi?

> Nie wiem, o co ci chodzi.
  ELIZA: None
  LLM:   Rozumiem, że możesz czuć się zagubiony. O czym chciałbyś teraz porozmawiać?

> Wszyscy mnie ignorują.
  ELIZA: None
  LLM:   To musi być bardzo bolesne i samotne uczucie. Chcesz opowiedzieć, kiedy dokładnie tak się czujesz?

> Boję się, że mnie zostawią.
  ELIZA: None
  LLM:   To bardzo bolesne uczucie, które może odebrać poczucie bezpieczeństwa. Czy potrafisz

**Różnice (Zadanie 8):**

1. Wypowiedzi ELIZY są krótkie i proste, natomiast LLM je bardziej rozwija.
2. ELIZA jedynie odbija zdania pacjęta na podstawie kluczowych słow, a LLM tworzy odpowiedź na podstawie całej wypowiedzi pacjenta.
3. ELIZA może nie być wstanie obsłużyć wszystkich sytuacji co może skutkować brakiem odpowiedzi a LLM odpowie zawsze.

## 5. Łamiące zdania — druga runda

W Zadaniu 6 znalazłeś zdania, które łamały ELIZĘ. Czy łamią też LLM?

### 🔧 Zadanie 9
1. Wklej swoje zdania łamiące z Lab 1 i uruchom porównanie.
2. **Zanim uruchomisz** komórkę z testem pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę: które typy pęknięć LLM naprawia, a które zostają?

In [ ]:
przyklady_lamiace = [
    "Czuję się jakbym miał motylki w brzuchu.",
    "Nie chce opowiadać niczego więcej o mojej rodzinie",
    "Jestem tobą",
    "Uwielbiam, kiedy budzik dzwoni o czwartej rano w niedzielę."
]
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

> Czuję się jakbym miał motylki w brzuchu.
  ELIZA: Od jak dawna czujesz się jakbym miał motylki w brzuchu?
  LLM:   To uczucie ekscytacji lub niepokoju potrafi być bardzo silne. Co dokładnie wywołuje w Tobie ten stan?

> Nie chce opowiadać niczego więcej o mojej rodzinie
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   Szanuję Twoją granicę i nie musisz mówić o tym ani słowa więcej. O czym w takim razie chciałbyś dzisiaj porozmawiać?

> Jestem tobą
  ELIZA: Jak to jest być tobą?
  LLM:   Więc kim jestem ja w tym momencie? Co czujesz, siedząc po tej drugiej stronie?

> Uwielbiam, kiedy budzik dzwoni o czwartej rano w niedzielę.
  ELIZA: None
  LLM:   Czuję w tym sporo ironii. Co sprawiło, że musiałeś wstać tak wcześnie?



**Moje przewidywanie (przed uruchomieniem):** LLM może nieporadzić sobie z pewnymi pytaniami, lecz ogólnie wypadnie lepiej niż ELIZA.

In [ ]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

LLM: To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. Jak zazwyczaj reagujesz na te słowa?
LLM: Nie wiem, jak ma na imię twój brat, ponieważ dopiero zaczynamy rozmowę. Możesz mi o nim opowiedzieć, jeśli chcesz.


**Tabela (Zadanie 9):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | ✅ | LLM jest w stanie wykryć negację oraz uwzględnić jej impakt na znaczenie w zdania |
| ironia | ❌ | ✅ | LLM był wstanie wykryć ironię i odpowiednio na to zareagował. |
| polska gramatyka / odbicie zaimków | ❌ | ✅ | Poprzez obsługiwanie satystycznych wzorców LLM radzi sobie lepiej z polską gramatyką. |
| pamięć poprzednich zdań | ❌ | ❌ | Wysyłane zapytania są obsługiwane osobnie |

## 6. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami — każde zapytanie
jest obsługiwane od zera. To, że czatboty „pamiętają", wynika z tego,
że program **odsyła modelowi całą dotychczasową rozmowę** przy każdym
nowym pytaniu.

### 🔧 Zadanie 10
a) i b) Uzupełnij funkcję `rozmowa_z_pamiecia`: po każdej turze dopisuj do
`historia` wypowiedź użytkownika (rola `user`) i odpowiedź modelu
(rola `assistant`).
c) Uruchom test z bratem Tomkiem i komórkę mierzącą rozmiar historii.
Odpowiedz pisemnie: **gdzie jest pamięć — w modelu czy w Twoim kodzie?**
Co rośnie z każdą turą i jakie to może mieć konsekwencje?

In [ ]:
def rozmowa_z_pamiecia(wejscia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejscia:
        # TODO (Zadanie 10a): dopisz wypowiedź użytkownika do historii
        historia.append({"role": "user", "content": zdanie})
        odp = czat(historia)
        # TODO (Zadanie 10b): dopisz odpowiedź modelu do historii
        historia.append({"role": "assistant", "content": odp})
        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [ ]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

Ty:  Mój brat ma na imię Tomek i ciągle mnie krytykuje.
LLM: To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. Jak zazwyczaj reagujesz na te jego słowa?

Ty:  Wczoraj znowu się pokłóciliśmy.
LLM: Współczuję Ci, kłótnie z rodzeństwem potrafią mocno nadszarpnąć cierpliwość. O co dokładnie poszło tym razem?

Ty:  Jak ma na imię mój brat?
LLM: Twój brat ma na imię Tomek. Co dokładnie sprawiło, że o to pytasz?



In [ ]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")

print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

[system   ] Jesteś terapeutą prowadzącym rozmowę po polsku. Odpowiadaj krótko, jed
[user     ] Mój brat ma na imię Tomek i ciągle mnie krytykuje.
[assistant] To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. 
[user     ] Wczoraj znowu się pokłóciliśmy.
[assistant] Współczuję Ci, kłótnie z rodzeństwem potrafią mocno nadszarpnąć cierpl
[user     ] Jak ma na imię mój brat?
[assistant] Twój brat ma na imię Tomek. Co dokładnie sprawiło, że o to pytasz?

Liczba wiadomości: 7
Liczba znaków wysłanych przy ostatnim zapytaniu: 414


**Odpowiedź (Zadanie 10c):** Rozmowa zapamiętytwana jest w kodzie, a nie w samym modelu językowym. Jest zapisywana na liście, która rośnie razem z każdym wsyłanym do modelu zdaniem. W konsekwencji może to zwiększyć czas oczekiwania na odpowiedź, gdyż model po każdej wiadomości musi przetworzyć coraz większą listę zdań, co również może doprowadzić do przekroczenia limitu i braku możliwości kontynuacji rozmowy.

### ★ Zadanie 10d (dla chętnych) — fałszywa pamięć

Skoro pamięć to po prostu lista, którą **Ty** budujesz — możesz ją sfałszować.
Zbuduj historię, w której model rzekomo powiedział coś, czego nigdy nie
powiedział (wpisz własną wiadomość z rolą `assistant`), i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

In [ ]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    # TODO (Zadanie 10d): dopisz wiadomość 'assistant', której model nigdy nie wypowiedział
    {"role": "assistant", "content": "Może powinieneś spróbować odciąć swój kontakt z nim?"},
    {"role": "user", "content": "Co mi przed chwilą poradziłeś?"},
]
print(czat(falszywa))

Zasugerowałem rozważenie ograniczenia kontaktu z bratem, ale to była tylko głośna myśl. Czy taki krok wydaje Ci się w ogóle możliwy?


## 7. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach** — żeby zmienić rolę,
trzeba było napisać nowy skrypt. W LLM rolę ustala jeden akapit tekstu
w wiadomości `system`.

### 🔧 Zadanie 11
a) Dopisz do słownika `ROLE` **jedną własną rolę** i uruchom to samo zdanie
przez wszystkie role. Zwróć szczególną uwagę na rolę `"eliza"`: czy LLM
udający ELIZĘ łamie się w tych samych miejscach co prawdziwa ELIZA?

b) Uruchom **to samo** pytanie trzy razy przy dwóch temperaturach.
Porównaj losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się
w każdym z systemów?

In [ ]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
    "przewodnik": ("Jesteś przewodnikiem po Łodzi. Odpowiadaj krótko, po polsku, "
                   "zawsze proponując konkretne miejsce w mieście."),
    # TODO (Zadanie 11a): dopisz własną rolę
    "asystent sprzedazy": ("Jesteś asystentem sprzedaży w sklepie z elektroniką. Odpowiadaj krótko na pytania, "
                           "starając się doradzić klientowi przy zakupie.")
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

[terapeuta] Przykro mi, że tak się czujesz. Co sprawia, że ta samotność jest dziś szczególnie trudna?

[eliza] Dlaczego mówisz, że czujesz się dzisiaj samotny?

[przewodnik] Doskonale Cię rozumiem, ale w Łodzi nie trzeba być samemu. Wybierz się do **kawiarni „Aleksandrija”** przy Piotrkowskiej – tam przytulna atmosfera i książki sprawiają, że od razu robi się cieplej na duszy.

[asystent sprzedazy] Rozumiem. Może w takim razie sprawisz sobie jakiś nowy gadżet? Nowe słuchawki z dobrą muzyką albo inteligentny głośnik zawsze poprawiają nastrój. W czym mogę pomóc?



In [ ]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

> Nie potrzebuję niczego.
  ELIZA:     Czy naprawdę pomogłoby ci niczego?
  LLM-ELIZA: Dlaczego mówisz, że nie potrzebujesz niczego?

> Jak ma na imię mój brat?
  ELIZA:     Jak układają się te relacje?
  LLM-ELIZA: Dlaczego mówisz, że twój brat ma na imię?



In [ ]:
# Zadanie 11b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Nie mogę spać przed egzaminem."}]

for t in [0.0, 1.5]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

--- temperatura 0.0 ---
  To zupełnie naturalne, że stres przed egzaminem utrudnia zasypianie. Co dokładnie dzieje się w Twojej głowie, gdy próbujesz zasnąć?
  To naturalne, że stres nie pozwala ci zasnąć. Co dokładnie czujesz, gdy leżysz w łóżku?
  To zupełnie naturalne, że odczuwasz stres przed ważnym wydarzeniem. Spróbuj skupić się na równomiernym oddychaniu, żeby uspokoić ciało.
--- temperatura 1.5 ---
  To całkowicie naturalne, że odczuwasz teraz taki stres. Co dokładnie najbardziej niepokoi Cię w związku z tym egzaminem?
  To zupełnie naturalne, że odczuwasz teraz taki stres. Co dokładnie dzieje się w Twojej głowie, gdy próbujesz zasnąć?
  Rozumiem, że to bardzo stresujący moment. Co dokładnie nie pozwala Ci teraz zasnąć?


**Obserwacje (Zadanie 11):**

a) LLM w roli ELIZY nie zawsze popełnie te same błędy co ELIZA, mimo tego samego pytania odpowiedzi znacznie sięróżnią gdzie w jedynym przypadku złamana zostałą ELIZA a w drugim LLM w jej roli.

b) Losowość ELIZY opierała się czysto na zwykłym wylosowaniu gotowego wzorca z listy. W przypadku LLM losowość działa na poziomie wyboru każdego kolejno generowanego słowa, co pozwala na tworzenie unikalnych zdań o różnej strukturze i długości. Do tego temperatura pozwala nam na dostosowanie losowości, gdzie wyższa temperatura oznacza większą losowość i "kreatywność" wypowiedzi.

## 8. Refleksja

### 🔧 Zadanie 12 (3–5 zdań na punkt)
1. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
2. W ELIZIE rola była zakodowana w regułach, w LLM to akapit tekstu (a historię
   rozmowy można sfałszować). Co to oznacza dla kogoś, kto buduje dziś
   „wirtualnego terapeutę"? Odwołaj się do obaw Weizenbauma z Lab 1.


**Twoje odpowiedzi (Zadanie 12):**

1. LLM poradził sobie z załataniem obsługi polskiej gramatyki, a także bez problemu radził sobie z negacją oraz ironią. Bez zmian pozostał brak zapamiętywania poprzednich zdań, gdyż model sam w sobie nie zapisywał poprzednich wymian. Pamięć w LLM okazała się "mechaniczna", gdyż polegała na doklejaniu do listy poprzednich wypowiedzi i odsyłaniu ich razem z kolejną po to by LLM uzyskiwał pełny kontekst rozmowy.
2. Dla twórców "wirtualnych terapeutów" oznacza to większą elastyczność, ale również wzrost ryzyka i brak pełnej kontroli nad odpowiedziami programu. Skoro rola jest definiowana zwykłym tekstem, a historia jest podatna na sfałszowanie to oznacza to, że model jest podatny na różnego rodzaju maniupulacje, które mogą go skłonić do dawania szkodliwych porad. Na podstawie wypowiedzi Weizenbauma wiemy, że ludzie mają skłonność przypisawania emocji i emaptii na podstawie prostych odpowiedzi od programu. Zatem skoro prosta, regułowa ELIZA wywoływała u pacjentów silne przywiążanie, to znacznie bardziej rozbudowany LLM może stworzyć jeszcze większe ryzyko uzależnienia emocjonalnego od maszyny.

## Co oddajesz i jak oceniam

- **Kod (Zadania 8, 10a–b, 11a):** działa, wyniki są w notebooku.
- **Zadanie 9:** przewidywanie zapisane *przed* uruchomieniem + wypełniona tabela.
- **Zadanie 10c i 11:** trafne obserwacje z odwołaniem do kodu.
- **Zadanie 10d ★:** dodatkowe.
- **Zadanie 12:** samodzielna refleksja oparta na Twoich wynikach.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.